# La voz de Chichito

Este cuaderno graba las frases de Chichito con la voz de los audios, usando **Chatterbox Multilingual** (Resemble AI, licencia MIT: se puede usar en el bar).

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4* (es gratis).

Después tocá ▶ en cada bloque, en orden. Todo tarda unos 10 a 15 minutos.

## 1. Instalar (2 a 3 minutos)
Después de instalar: **Entorno de ejecución → Reiniciar sesión** y seguí desde el paso 2.

In [ ]:
!pip install -q chatterbox-tts
!pip install -q torchvision==0.21.0   # que haga juego con el torch que instala chatterbox
!apt-get -qq install -y ffmpeg > /dev/null
# Los avisos rojos de 'dependency conflicts' son normales.
# IMPORTANTE: después de esto, menú Entorno de ejecución → Reiniciar sesión, y seguí desde el paso 2.

## 2. Subir los archivos
Elegí **referencia-14s.wav** y el archivo de frases: **frases.json** la primera vez, o **frases-nuevas.json** cuando son solo las nuevas (no hace falta cambiarle el nombre).

In [ ]:
from google.colab import files
import os, glob
subidos = files.upload()
print('Subido:', list(subidos))
# El archivo de frases que se subió recién (si había uno viejo con el mismo nombre, Colab le agrega (1), (2)…)
FRASES = next((n for n in subidos if n.endswith('.json')), None) or max(glob.glob('frases*.json'), key=os.path.getmtime)
print('Frases a grabar:', FRASES)

## 3. Cargar el modelo

In [ ]:
import torch, torchaudio as ta
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
try:
    modelo = ChatterboxMultilingualTTS.from_pretrained(device='cuda', t3_model='v3')
except TypeError:
    modelo = ChatterboxMultilingualTTS.from_pretrained(device='cuda')
REF = 'referencia-14s.wav'
print('Modelo listo')

## 4. Probar antes de grabar todo
Escuchá las dos pruebas. Si suena muy rápido, bajá `CFG` a 0.3. Si suena chato, subí `EXPRESION` a 0.7. Si no se parece, probá con `REF = 'referencia-completa.wav'` (subila también).

In [ ]:
from IPython.display import Audio, display
EXPRESION = 0.5
CFG = 0.5
def grabar(texto):
    return modelo.generate(texto, language_id='es', audio_prompt_path=REF, exaggeration=EXPRESION, cfg_weight=CFG)
for t in ['¡Hola! Soy Chichito. Tocame cuando necesites una mano.', 'Todavía no abriste la caja de hoy. Cargá el efectivo inicial así el cierre da justo.']:
    w = grabar(t)
    display(Audio(w.squeeze(0).cpu().numpy(), rate=modelo.sr))

## 5. Grabar todas las frases y bajar el zip
Cuando termine se baja **voz-chichito.zip**. Mandámelo y lo pongo en la app.

In [ ]:
import json, os, shutil, subprocess, zipfile
frases = json.load(open(FRASES, encoding='utf-8'))
print(f'Voy a grabar {len(frases)} frases de {FRASES}')
shutil.rmtree('voz', ignore_errors=True)   # que no se cuelen audios de una vez anterior
os.makedirs('voz')
indice = {}
for i, f in enumerate(frases, 1):
    w = grabar(f['texto'])
    ta.save('tmp.wav', w.cpu(), modelo.sr)
    subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-i', 'tmp.wav', '-ac', '1', '-b:a', '64k', f"voz/{f['id']}.mp3"], check=True)
    indice[f['id']] = f"{f['id']}.mp3"
    print(f'{i}/{len(frases)}', f['texto'][:60])
json.dump(indice, open('voz/indice.json', 'w'))
if os.path.exists('voz-chichito.zip'): os.remove('voz-chichito.zip')
with zipfile.ZipFile('voz-chichito.zip', 'w') as z:
    for n in os.listdir('voz'): z.write(f'voz/{n}', n)
files.download('voz-chichito.zip')